# Notebook 08: Activity-Adaptive Anatomical Graph Neural Network (Adaptive GNN)

**Objective**: Train and evaluate the core research model — the Activity-Adaptive Anatomical Graph Neural Network (`ActivityAdaptiveAnatomicalGNN`).

### Core Research Hypothesis:
Different activities (walking, sitting, standing, reaching) induce dynamic, task-dependent functional coupling between limbs. An adaptive graph formulation where edge weights depend on the active input window $X$ should capture these relationships more effectively than a fixed, static skeletal prior.

### Dynamic Graph Formulation:
$$A_{\text{dyn}} = (1 - \lambda) \tilde{A}_{\text{fixed}} + \lambda A_{\text{adapt}}(h_0)$$
where $\lambda = \sigma(w_\lambda)$ balances the documented anatomical skeletal prior $\tilde{A}_{\text{fixed}}$ with the learned pairwise attention affinity $A_{\text{adapt}}$.

> **Compute Rule**: Full training requires CUDA GPU (Colab). Set `EXPERIMENT_MODE = 'DEBUG'` for 1-batch CPU verification.

## 1. Configure Experiment Mode

In [ ]:
# Choose: 'DEBUG' (1-epoch CPU/GPU check), 'FAST' (5 epochs GPU), or 'FULL' (50 epochs GPU)
EXPERIMENT_MODE = "DEBUG"

import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

current = Path.cwd()
REPO_ROOT = current.parent if current.name == "notebooks" else current
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.api as api
from src.features.body_regions import BODY_REGIONS
env_info = api.setup_project()

## 2. Load Processed Sliding Windows & DataLoaders

In [ ]:
subset = api.load_preprocessed_subset("opportunity_locomotion_subset.npz")
X_train, y_train = subset["X_train"], subset["y_train"]
X_val, y_val = subset["X_val"], subset["y_val"]
X_test, y_test = subset["X_test"], subset["y_test"]

if EXPERIMENT_MODE == "DEBUG":
    X_train, y_train = X_train[:64], y_train[:64]
    X_val, y_val = X_val[:32], y_val[:32]
    X_test, y_test = X_test[:32], y_test[:32]

train_loader, val_loader, test_loader = api.create_dataloaders(
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    batch_size=64 if EXPERIMENT_MODE != "DEBUG" else 16,
)
print(f"DataLoaders: {len(train_loader)} train batches, {len(val_loader)} val batches.")

## 3. Instantiate Activity-Adaptive Anatomical GNN

In [ ]:
adaptive_gnn = api.build_adaptive_gnn(
    num_classes=5,
    sequence_length=30,
    node_dim=64,
    gcn_hidden_dim=64,
    dropout=0.2
)

total_params = sum(p.numel() for p in adaptive_gnn.parameters() if p.requires_grad)
print(f"ActivityAdaptiveAnatomicalGNN Initialized! Trainable parameters: {total_params:,}")

## 4. Train Model (Automatic GPU/CUDA Guard)

In [ ]:
best_adaptive_gnn, history = api.train_model(
    model=adaptive_gnn,
    train_loader=train_loader,
    val_loader=val_loader,
    mode=EXPERIMENT_MODE,
    lr=1e-3,
    experiment_name="adaptive_anatomical_gnn"
)
print("Training Complete!")

## 5. Evaluate on Held-Out Test Set

In [ ]:
test_metrics = api.evaluate_model(best_adaptive_gnn, test_loader)

print("\n" + "=" * 55)
print(" ACTIVITY-ADAPTIVE ANATOMICAL GNN EVALUATION")
print("=" * 55)
print(f"Accuracy    : {test_metrics['accuracy'] * 100:.2f}%")
print(f"Macro F1    : {test_metrics['macro_f1'] * 100:.2f}%")
print(f"Weighted F1 : {test_metrics['weighted_f1'] * 100:.2f}%")
print("=" * 55)

## 6. Inspect & Visualize Learned Dynamic Adjacency Matrices
Extract and plot the learned inter-region edge weights $A_{\text{dyn}}$ for sample activity windows.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_adaptive_gnn.eval()

sample_x = torch.from_numpy(X_test[:4]).float().to(device)
with torch.no_grad():
    logits, dynamic_adj = best_adaptive_gnn(sample_x, return_attention=True)

adj_matrices = dynamic_adj.cpu().numpy()
region_names = [r.replace("_", " ").title() for r in BODY_REGIONS]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for i in range(2):
    sns.heatmap(
        adj_matrices[i],
        annot=True, fmt=".2f", cmap="YlGnBu",
        xticklabels=region_names, yticklabels=region_names,
        ax=axes[i], vmin=0.0, vmax=0.6
    )
    axes[i].set_title(f"Dynamic Adjacency A_dyn (Window #{i})")
    axes[i].set_xlabel("Target Region")
    axes[i].set_ylabel("Source Region")

plt.tight_layout()
plt.show()